In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

In [ ]:
path = "path/to/HanHoJu_2.xlsx"

In [ ]:
drop_cols = [
    "Ref_ID",
    "JV_reverse_scan_Voc",
    "JV_reverse_scan_Jsc",
    "JV_reverse_scan_FF"
]


In [ ]:
target_col = "JV_reverse_scan_PCE"
K = 100
subsample_ratio = 0.8
seed0 = 42


In [ ]:
df = pd.read_excel(path, sheet_name=0)

drop_cols = [c for c in drop_cols if c in df.columns]
df_ = df.drop(columns=drop_cols)

assert target_col in df_.columns

y = df_[target_col]
X = df_.drop(columns=[target_col])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [ ]:
num = X_train.select_dtypes(include=["int64", "float64"]).copy()
num = num.replace([np.inf, -np.inf], np.nan)
num = num.fillna(num.median(numeric_only=True))

cols = num.columns.tolist()
X_full = num.to_numpy(dtype=float)
n_full, p = X_full.shape

tri = np.triu_indices(p, k=1)

C_full_pearson = num.corr(method="pearson").to_numpy()
C_full_spearman = num.corr(method="spearman").to_numpy()


def center_distance_matrix(D):
    row_mean = D.mean(axis=1, keepdims=True)
    col_mean = D.mean(axis=0, keepdims=True)
    grand_mean = D.mean()
    return D - row_mean - col_mean + grand_mean


def centered_dist_mats(Xn):
    mats = []
    for j in range(Xn.shape[1]):
        x = Xn[:, j].astype(float).reshape(-1, 1)
        D = np.abs(x - x.T)
        A = center_distance_matrix(D)
        mats.append(A)
    return mats


def dcor_matrix(Xn):
    n, p = Xn.shape
    A = centered_dist_mats(Xn)
    dcor = np.eye(p, dtype=float)
    norms = np.array([np.sqrt((Aj * Aj).mean()) for Aj in A], dtype=float)

    for i in range(p):
        for j in range(i + 1, p):
            if norms[i] <= 0 or norms[j] <= 0:
                v = 0.0
            else:
                dcov = np.sqrt((A[i] * A[j]).mean())
                v = float(dcov / np.sqrt(norms[i] * norms[j]))
            dcor[i, j] = v
            dcor[j, i] = v

    return dcor


C_full_dcor = dcor_matrix(X_full)


rng = np.random.default_rng(seed0)
n_sub = int(round(n_full * subsample_ratio))

dist_pearson = np.empty(K, dtype=float)
dist_spearman = np.empty(K, dtype=float)
dist_dcor = np.empty(K, dtype=float)

for k in range(K):
    idx = rng.choice(n_full, size=n_sub, replace=False)
    sub = num.iloc[idx]

    Ck_p = sub.corr(method="pearson").to_numpy()
    Ck_s = sub.corr(method="spearman").to_numpy()
    Ck_d = dcor_matrix(sub.to_numpy(dtype=float))

    dist_pearson[k] = np.mean(np.abs(Ck_p[tri] - C_full_pearson[tri]))
    dist_spearman[k] = np.mean(np.abs(Ck_s[tri] - C_full_spearman[tri]))
    dist_dcor[k] = np.mean(np.abs(Ck_d[tri] - C_full_dcor[tri]))


def summarize(name, arr):
    mu = float(np.mean(arr))
    sd = float(np.std(arr, ddof=1))
    q95 = float(np.quantile(arr, 0.95))
    print(f"{name}: mean={mu:.6f}, std={sd:.6f}, 95th={q95:.6f}")
    return mu, sd, q95


print(f"n_full={n_full}, n_sub={n_sub}, p={p}, K={K}")

mu_p, sd_p, q95_p = summarize("Pearson mean|Δr|", dist_pearson)
mu_s, sd_s, q95_s = summarize("Spearman mean|Δr|", dist_spearman)
mu_d, sd_d, q95_d = summarize("dCor mean|Δr|", dist_dcor)

avg_q95 = float(np.mean([q95_p, q95_s, q95_d]))
print(f"Average 95th threshold: {avg_q95:.6f}")


out = pd.DataFrame({
    "pearson": dist_pearson,
    "spearman": dist_spearman,
    "dcor": dist_dcor
})

out.to_csv("bootstrap_natural_variability_meanAbsDelta.csv", index=False)